In [8]:
import os
import pandas as pd
import numpy as np
import load_dotenv
import io
import re
import ast
import contextlib
import types
import unittest
import importlib
import sys
from openai import OpenAI
from datasets import load_dataset

In [9]:
load_dotenv.load_dotenv()

# OPEN_AI_API = os.getenv("OPEN_AI_API")
OPEN_AI_API = os.getenv("OPEN_AI_API_v2")
if OPEN_AI_API is None:
    raise ValueError("OPEN_AI_API environment variable not set")
else:
    print("API key loaded successfully")
    
    
MODEL_NAME = "gpt-5-nano"  # or "gpt-4o" or "gpt-5-nano"

API key loaded successfully


# 1. Load sample

In [10]:
client = OpenAI(api_key=OPEN_AI_API)

In [11]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Who is the current prime minister of VietNam?"},
    ],
)

output = response.choices[0].message.content

print("Response from OpenAI:")
print(output)

Response from OpenAI:
Phạm Minh Chính. He has served as Vietnam’s Prime Minister since April 5, 2021, and was still in office as of June 2024. If you need the absolutely latest status, please check a current news source.


# 2. Test complete code

In [12]:
code_description = "write a function to compute the sum of python list"
function_signature = """
def sum_of_list(input_list):
"""
constraints = """
Output only a complete and valud Python code for a this function. 
Do not add more explanations or surrounding text and Do not change the provided function signature.
Wrap your output strictly between the markers:
<code>
... your code ...
</code>
"""

input_prompt = f"""write a complete python function
based on the following description:\n{code_description}.\n
The function signature is:\n{function_signature}\n.
with the following constraints:\n{constraints}
"""

In [13]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are an expert in Python."},
        {"role": "user", "content": input_prompt},
    ],
)

output = response.choices[0].message.content

print("Response from OpenAI:")
print(output)

Response from OpenAI:
<code>
def sum_of_list(input_list):
    if input_list is None:
        return 0
    total = 0
    for item in input_list:
        if isinstance(item, (list, tuple)):
            total += sum_of_list(item)
        elif item is None:
            continue
        else:
            total += item
    return total
</code>


In [14]:
def extract_function(llm_text):
    # 1) Grab text between <code>...</code>
    m = re.search(r"<code>\s*(.*?)\s*</code>", llm_text, flags=re.S|re.M)
    if not m:
        raise ValueError("No <code> block found")
    code = m.group(1)

    # 2) Optionally, if the model sometimes adds backticks, strip them
    code = re.sub(r"^```(?:python)?\s*|\s*```$", "", code.strip())

    return code

In [15]:
completed_code = extract_function(output)
print(f"The complete code:\n")
print(completed_code)

The complete code:

def sum_of_list(input_list):
    if input_list is None:
        return 0
    total = 0
    for item in input_list:
        if isinstance(item, (list, tuple)):
            total += sum_of_list(item)
        elif item is None:
            continue
        else:
            total += item
    return total


# 3. Test code generation on BigCodeBench benchmark

In [16]:
dataset = load_dataset("bigcode/bigcodebench", split="v0.1.4")

df = pd.DataFrame(dataset)

print(f"Shape of DataFrame: {df.shape}")
df.sample(1)

Shape of DataFrame: (1140, 9)


,task_id,complete_prompt,instruct_prompt,canonical_solution,code_prompt,test,entry_point,doc_struct,libs
1073,BigCodeBench/1073,import time\nimport matplotlib.pyplot as plt\n...,Parses a list of time strings and plots a hist...,"try:\n seconds = [time.strptime(ts,...",import time\nimport matplotlib.pyplot as plt\n...,import unittest\nimport matplotlib.pyplot as p...,task_func,"{""description"": [""Parses a list of time string...","['matplotlib', 'time']"


In [17]:
idx = np.random.randint(0, len(df))

complete_prompt = df.loc[idx, "complete_prompt"]
test_case = df.loc[idx, "test"]
list_libs = df.loc[idx, 'libs']
list_libs = ast.literal_eval(list_libs)  # Convert list_libs from string representation to actual list
print(f"List libs: {list_libs}")

List libs: ['numpy', 'collections', 'matplotlib', 'scipy']


In [18]:
print(test_case)

import unittest
import pandas as pd
from collections import Counter
import matplotlib
class TestCases(unittest.TestCase):
    def _check_plot(self, ax):
        self.assertIsInstance(ax, plt.Axes)
        self.assertEqual(ax.get_title(), "Distribution")
        self.assertEqual(ax.get_xlabel(), "Value")
        self.assertEqual(ax.get_ylabel(), "Frequency")
    def test_case_1(self):
        # Basic case - no repeated value
        df = pd.DataFrame({"value": [1, 2, 3, 4, 5]})
        counter, ax = task_func(df)
        self._check_plot(ax)
        self.assertEqual(counter, Counter())
    def test_case_2(self):
        # Basic case - all repeated values
        df = pd.DataFrame({"value": [1, 1, 1, 1, 1]})
        counter, ax = task_func(df)
        self._check_plot(ax)
        self.assertEqual(counter, Counter({1: 5}))
    def test_case_3(self):
        # Basic case - test empty
        df = pd.DataFrame({"value": []})
        counter, ax = task_func(df)
        self.assertIsInstance(

In [19]:
# Form the input prompt
input_prompt = f"""write a complete python function
based on the following description:\n{complete_prompt}.\n
with the following constraints:\n{constraints}
"""

print("Input Prompt:\n", input_prompt)

Input Prompt:
 write a complete python function
based on the following description:
import numpy as np
from collections import Counter
from scipy.stats import norm
import matplotlib.pyplot as plt


def task_func(df, bins=4):
    """
    Identify and count duplicate values in a DataFrame's 'value' column.
    This function also plots a histogram for all values in the 'value' column
    and overlays a normal distribution curve on the histogram.

    Parameters:
    df (pd.DataFrame): DataFrame containing a numeric 'value' column. If empty,
                       the function will return empty Counter and an empty plot.
    bins (int, optional): Number of bins for the histogram. Defaults to 4.

    Returns:
    tuple: A tuple containing:
        - Counter: A Counter object with the count of each duplicate value.
        - Axes: A matplotlib.axes.Axes object that represents the plot
                of the histogram with the 'value' column data. If applicable,
                a normal distr

In [20]:
response = client.chat.completions.create(
    model="gpt-4o",  # or "gpt-4o"
    messages=[
        {"role": "system", "content": "You are a Python expert."},
        {"role": "user", "content": input_prompt},
    ],
)

generated_code = response.choices[0].message.content

print("Response from OpenAI:")
print(generated_code)

Response from OpenAI:
<code>
import numpy as np
from collections import Counter
from scipy.stats import norm
import matplotlib.pyplot as plt
import pandas as pd

def task_func(df, bins=4):
    if df.empty or 'value' not in df:
        return Counter(), plt.gca()

    values = df['value']
    duplicates = [item for item, count in Counter(values).items() if count > 1]
    counter = Counter({item: count for item, count in Counter(values).items() if item in duplicates})

    fig, ax = plt.subplots()
    n, bins, patches = ax.hist(values, bins=bins, color='green', alpha=0.6, density=True)

    if len(values) > 0:
        mu, std = norm.fit(values)
        xmin, xmax = plt.xlim()
        x = np.linspace(xmin, xmax, 100)
        p = norm.pdf(x, mu, std)
        ax.plot(x, p, 'k', linewidth=2)

    ax.set_title('Distribution')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    
    return counter, ax
</code>


In [21]:
# Test case
full_code = extract_function(generated_code)

print("Full Code to Execute:\n")
print(full_code)

Full Code to Execute:

import numpy as np
from collections import Counter
from scipy.stats import norm
import matplotlib.pyplot as plt
import pandas as pd

def task_func(df, bins=4):
    if df.empty or 'value' not in df:
        return Counter(), plt.gca()

    values = df['value']
    duplicates = [item for item, count in Counter(values).items() if count > 1]
    counter = Counter({item: count for item, count in Counter(values).items() if item in duplicates})

    fig, ax = plt.subplots()
    n, bins, patches = ax.hist(values, bins=bins, color='green', alpha=0.6, density=True)

    if len(values) > 0:
        mu, std = norm.fit(values)
        xmin, xmax = plt.xlim()
        x = np.linspace(xmin, xmax, 100)
        p = norm.pdf(x, mu, std)
        ax.plot(x, p, 'k', linewidth=2)

    ax.set_title('Distribution')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    
    return counter, ax


# 3. Run whole dataset

In [ ]:
out_generated_code = []

for idx in range(len(df)):
    if idx % 100 == 0:
        print(f"Processing index: {idx}/{len(df)}")
    
    complete_prompt = df.loc[idx, "complete_prompt"]
    test_case = df.loc[idx, "test"]
    list_libs = df.loc[idx, 'libs']
    list_libs = ast.literal_eval(list_libs)  # Convert list_libs from
    
    input_prompt = f"""write a complete python function
    based on the following description:\n{complete_prompt}.\n
    with the following constraints:\n{constraints}
    """
    
    response = client.chat.completions.create(
        model=MODEL_NAME,  # or "gpt-4o"
        messages=[
            {"role": "system", "content": "You are a Python expert."},
            {"role": "user", "content": input_prompt},
        ],
    )
    
    generated_code = response.choices[0].message.content
    
    # extract function
    full_code = extract_function(generated_code)
    
    out_generated_code.append({
        "idx": idx,
        "complete_prompt": complete_prompt,
        "generated_code": full_code,
        "test_case": test_case,
        "libs": list_libs
    })


out_generated_code_df = pd.DataFrame(out_generated_code)
out_generated_code_df.to_csv(f"generated_code_{MODEL_NAME}.csv", index=False)

Processing index: 0/1140
